In [1]:
import numpy as np
import numpy as np
import matplotlib.pyplot as plt
import time

from matplotlib.colors import LinearSegmentedColormap
from scipy.optimize import minimize
from scipy.optimize import curve_fit
from scipy.interpolate import interp1d
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import efficient_su2
from joblib import Parallel, delayed


In [2]:
# Define parameters

N = 6          # number of qubits
J = 1.0        # coupling strength
reps = 2
tol = 0.1 
epsilon_N = 1e-6 * (4 / N) ** 4    # symmetry-breaking field

# Define h values
h_values = np.concatenate([
    np.linspace(0.2, 0.8, 8),      # Ordered region 8 points
    np.linspace(0.8, 1.2, 10),     # Critical region 10 points 
    np.linspace(1.2, 3.0, 7)       # Disordered region 7 points
])    


# Finite-size scaling parameters (exact values for 1D TFIM)
N_values = [4, 6, 8, 10]    # system sizes to sweep
n_restarts = 1              # number of restarts per point
num_processes = 4           # number of processes for parallelization
beta_exp   = 0.125
nu_exp     = 1.0

In [ ]:
def build_tfim_hamiltonian(N: int, J: float, h: float,
                            epsilon: float = 1e-4) -> SparsePauliOp:

    pauli_terms = []
    for i in range(N - 1):
        pauli_str = ["I"] * N
        pauli_str[i]     = "Z"
        pauli_str[i + 1] = "Z"
        pauli_terms.append(("".join(reversed(pauli_str)), -J))
    for i in range(N):
        pauli_str = ["I"] * N
        pauli_str[i] = "X"
        pauli_terms.append(("".join(reversed(pauli_str)), -h))
    # Small longitudinal field to break Z2 symmetry
    for i in range(N):
        pauli_str = ["I"] * N
        pauli_str[i] = "Z"
        pauli_terms.append(("".join(reversed(pauli_str)), -epsilon))
    return SparsePauliOp.from_list(pauli_terms)


def build_ansatz(N: int, reps: int = 2):
    return efficient_su2(num_qubits=N, reps=reps, entanglement="linear")


def build_magnetisation_op(N: int) -> SparsePauliOp:
    pauli_terms = []
    for i in range(N):
        pauli_str = ["I"] * N
        pauli_str[i] = "Z"
        pauli_terms.append(("".join(reversed(pauli_str)), 1 / N))
    return SparsePauliOp.from_list(pauli_terms)


def exact_ground_state_energy(N: int, J: float, h: float) -> float:
    H_matrix = build_tfim_hamiltonian(N, J, h).to_matrix()
    return float(np.linalg.eigvalsh(H_matrix)[0])


def run_vqe(N: int, J: float, h: float, reps: int = 2,
            seed: int = 42, warm_start_params: np.ndarray = None):

    hamiltonian = build_tfim_hamiltonian(N, J, h)
    ansatz      = build_ansatz(N, reps)
    estimator   = StatevectorEstimator()

    def cost_fn(params):
        pub    = (ansatz, hamiltonian, params)
        result = estimator.run([pub]).result()
        return float(np.real(result[0].data.evs))

    # Determine if near critical point
    near_critical = abs(h / J - 1.0) < 0.3
    
    # Adaptive tolerances - Ordered/disordered regions converge faster with looser tolerances
    if near_critical:
        ftol = 1e-9  # Tight tolerance near critical point
        gtol = 1e-6
    else:
        ftol = 1e-6  # Looser tolerance away from critical point
        gtol = 1e-5
    
    # Adaptive maxiter
    if near_critical:
        maxiter = 500  # More iterations near critical point
    else:
        maxiter = 300  # Fewer iterations away from critical point

    attempts = n_restarts if (near_critical and warm_start_params is None) else 1
    

    # If warm start provided, use it as the sole starting point
    if warm_start_params is not None:
        starting_points = [warm_start_params]
    else:
        starting_points = [
            np.random.default_rng(seed + i).uniform(-np.pi, np.pi, ansatz.num_parameters)
            for i in range(attempts)
        ]

    best_energy = np.inf
    best_params = None

    for init_params in starting_points:
        result = minimize(
            cost_fn,
            init_params,
            method="L-BFGS-B",           # gradient-based, fast on statevector
            options={"maxiter": maxiter, "ftol": ftol, "gtol": gtol}
        )
        if result.fun < best_energy:
            best_energy = result.fun
            best_params = result.x

    return best_energy, best_params


def classify_phase(h: float, J: float, tol: float) -> str:
    ratio = h / J
    if abs(ratio - 1.0) < tol:
        return " CRITICAL "
    elif ratio < 1.0:
        return " ORDERED  "
    else:
        return "DISORDERED"


# Helper function for parallel processing
def process_single_h(h_index_tuple, N, J, h_values, reps, prev_params_dict, M_op, tol):
    """
    Process a single h value in parallel.
    Returns: (h_index, h, E_exact, E_vqe, M, params)
    """
    h_index, h = h_index_tuple
    phase = classify_phase(h, J, tol)
    print(f"[{h_index+1}/{len(h_values)}]  h/J = {h/J:.3f}  [{phase}]", end="  ")

    E_exact = exact_ground_state_energy(N, J, h)

    ansatz = build_ansatz(N, reps)

    # Use warm-start params if available from previous point
    prev_params = prev_params_dict.get(h_index - 1, None)
    E_vqe, params = run_vqe(N, J, h, reps=reps, warm_start_params=prev_params)

    # Calculate magnetisation
    estimator = StatevectorEstimator()
    pub    = (ansatz, M_op, params)
    result = estimator.run([pub]).result()
    M      = abs(float(np.real(result[0].data.evs)))

    print(f"E_exact = {E_exact:.4f}   E_vqe = {E_vqe:.4f}   "
          f"err = {abs(E_vqe - E_exact):.4f}   |M| = {M:.4f}")

    return (h_index, h, E_exact, E_vqe, M, params)


def sweep_phase_diagram(N: int, J: float, h_values: np.ndarray, reps: int = 2):
    """
    Returns (vqe_energies, exact_energies, magnetisations).
    Sequential sweep with warm starting.
    """
    vqe_energies   = []
    exact_energies = []
    magnetisations = []

    M_op         = build_magnetisation_op(N)
    estimator    = StatevectorEstimator()
    prev_params  = None

    for i, h in enumerate(h_values):
        phase = classify_phase(h, J, tol)
        print(f"[{i+1}/{len(h_values)}]  h/J = {h/J:.3f}  [{phase}]", end="  ")

        E_exact = exact_ground_state_energy(N, J, h)
        exact_energies.append(E_exact)

        ansatz        = build_ansatz(N, reps)
        E_vqe, params = run_vqe(N, J, h, reps=reps,
                                 warm_start_params=prev_params)
        prev_params   = params
        vqe_energies.append(E_vqe)

        pub    = (ansatz, M_op, params)
        result = estimator.run([pub]).result()
        M      = abs(float(np.real(result[0].data.evs)))
        magnetisations.append(M)

        print(f"E_exact = {E_exact:.4f}   E_vqe = {E_vqe:.4f}   "
              f"err = {abs(E_vqe - E_exact):.4f}   |M| = {M:.4f}")

    return (np.array(vqe_energies),
            np.array(exact_energies),
            np.array(magnetisations))


def run_single_N_sweep(N, J, h_values, reps):
    n_reps = reps if N <= 6 else reps + 1
    print(f"\n{'='*50}\nRunning sweep for N={N}, reps={n_reps}\n{'='*50}")
    start = time.time()
    vqe_energies, exact_energies, magnetisations = sweep_phase_diagram(
        N, J, h_values, reps=n_reps
    )
    print(f"Time for N={N}: {time.time() - start:.1f}s")
    return N, vqe_energies, exact_energies, magnetisations


def run_finite_size_sweep(N_values, J, h_values, reps=2):
    results = {}
    for N in N_values:
        n_reps = reps if N <= 6 else reps + 1
        print(f"\n{'='*50}")
        print(f"Running sweep for N={N}, reps={n_reps}")
        print(f"{'='*50}")
        start = time.time()

        vqe_energies, exact_energies, magnetisations = sweep_phase_diagram(
            N, J, h_values, reps=n_reps
        )

        elapsed = time.time() - start
        print(f"Time for N={N}: {elapsed:.1f}s")

        results[N] = {
            "vqe_energies"  : vqe_energies,
            "exact_energies": exact_energies,
            "magnetisations": magnetisations
        }
    return results

results = run_finite_size_sweep(N_values, J, h_values, reps=reps)


Running sweep for N=4, reps=2
[1/25]  h/J = 0.200  [ ORDERED  ]  E_exact = -3.0618   E_vqe = -3.0603   err = 0.0015   |M| = 0.9875
[2/25]  h/J = 0.286  [ ORDERED  ]  E_exact = -3.1294   E_vqe = -3.1224   err = 0.0070   |M| = 0.9745
[3/25]  h/J = 0.371  [ ORDERED  ]  

In [ ]:
# Plot the measurements

colors = ["steelblue", "teal", "coral", "purple"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot Magnetisation vs h/J for increasing N
for (N, data), color in zip(results.items(), colors):
    axes[0].plot(h_values / J, data["magnetisations"],
                 "o-", label=f"N={N}", color=color)
axes[0].axvline(x=1.0, color="gray", linestyle=":", label="h/J = 1")
axes[0].set_xlabel("h / J")
axes[0].set_ylabel("|⟨M⟩|")
axes[0].set_title("Magnetisation vs h/J for increasing N")
axes[0].legend()

# Plot VQE error
for (N, data), color in zip(results.items(), colors):
    axes[1].plot(h_values / J,
                 np.abs(data["vqe_energies"] - data["exact_energies"]),
                 "o-", label=f"N={N}", color=color)
axes[1].axvline(x=1.0, color="gray", linestyle=":")
axes[1].set_xlabel("h / J")
axes[1].set_ylabel("|E_VQE - E_exact|")
axes[1].set_title("VQE error vs system size")
axes[1].legend()

plt.tight_layout()
plt.savefig("tfim_finite_size.png", dpi=150)
plt.show()

# Data collapse
fig, ax = plt.subplots(figsize=(8, 5))
for (N, data), color in zip(results.items(), colors):
    x_scaled = (h_values / J - 1.0) * (N ** (1.0 / nu_exp))
    y_scaled = data["magnetisations"] * (N ** (beta_exp / nu_exp))
    ax.plot(x_scaled, y_scaled, "o-", label=f"N={N}", color=color)
ax.set_xlabel(r"$(h/J - 1) \cdot N^{1/\nu}$")
ax.set_ylabel(r"$|\langle M \rangle| \cdot N^{\beta/\nu}$")
ax.set_title("Finite-size scaling collapse (β=1/8, ν=1)")
ax.axvline(x=0, color="gray", linestyle=":", label="Critical point")
ax.legend()
ax.set_xlim(-8, 8)
plt.tight_layout()
plt.savefig("tfim_data_collapse.png", dpi=150)
plt.show()



for (N, data), color in zip(results.items(), colors):
    x_scaled = (h_values / J - 1.0) * (N ** (1.0 / nu_exp))

    # VQE magnetisation collapse
    y_vqe = data["magnetisations"] * (N ** (beta_exp / nu_exp))
    axes[0].plot(x_scaled, y_vqe, "o-", label=f"N={N}", color=color)

    # Exact magnetisation collapse — compute from exact ground state
    exact_mags = []
    for h in h_values:
        H_matrix  = build_tfim_hamiltonian(N, J, h).to_matrix()
        eigenvalues, eigenvectors = np.linalg.eigh(H_matrix)
        ground_state = eigenvectors[:, 0]

        M_op     = build_magnetisation_op(N)
        M_matrix = M_op.to_matrix()
        M_exact  = abs(float(np.real(ground_state.conj() @ M_matrix @ ground_state)))
        exact_mags.append(M_exact)

    y_exact = np.array(exact_mags) * (N ** (beta_exp / nu_exp))
    axes[1].plot(x_scaled, y_exact, "o-", label=f"N={N}", color=color)

for ax, title in zip(axes, ["VQE collapse", "Exact collapse"]):
    ax.axvline(x=0, color="gray", linestyle=":", label="Critical point")
    ax.set_xlabel(r"$(h/J - 1) \cdot N^{1/\nu}$")
    ax.set_ylabel(r"$|\langle M \rangle| \cdot N^{\beta/\nu}$")
    ax.set_title(f"Finite-size scaling collapse — {title} (β=1/8, ν=1)")
    ax.legend()
    ax.set_xlim(-8, 8)   

plt.tight_layout()
plt.savefig("tfim_collapse_comparison.png", dpi=150)
plt.show()

plt.close('all')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

In [ ]:
def exact_magnetisation(N: int, J: float, h: float) -> float:
    """
    Exact magnetisation with symmetry-breaking field to select ordered state.
    """
    H_matrix             = build_tfim_hamiltonian(N, J, h).to_matrix()
    eigenvalues, eigvecs = np.linalg.eigh(H_matrix)
    ground_state         = eigvecs[:, 0]
    M_matrix             = build_magnetisation_op(N).to_matrix()
    M_val                = ground_state.conj() @ M_matrix @ ground_state
    return abs(float(np.real(M_val)))

def exact_energy_gap(N: int, J: float, h: float) -> float:
    """
    Compute the energy gap between ground state and first excited state.
    Gap closes at the critical point h/J = 1.
    """
    H_matrix   = build_tfim_hamiltonian(N, J, h).to_matrix()
    eigenvalues = np.linalg.eigvalsh(H_matrix)
    return float(eigenvalues[1] - eigenvalues[0])

def exact_benchmark_sweep(N_values, J, h_values):
    """
    Compute exact energies, magnetisations, and energy gaps
    for all N and h values.
    """
    benchmark = {}

    for N in N_values:
        print(f"Benchmarking N={N}...")
        energies      = []
        magnetisations = []
        gaps           = []

        for h in h_values:
            energies.append(exact_ground_state_energy(N, J, h))
            magnetisations.append(exact_magnetisation(N, J, h))
            gaps.append(exact_energy_gap(N, J, h))

        benchmark[N] = {
            "energies"      : np.array(energies),
            "magnetisations": np.array(magnetisations),
            "gaps"          : np.array(gaps)
        }

    return benchmark

benchmark = exact_benchmark_sweep(N_values, J, h_values)


fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ["steelblue", "teal", "coral", "purple"]

for (N, data), color in zip(results.items(), colors):
    b = benchmark[N]

    # Energy comparison
    axes[0].plot(h_values / J, b["energies"],
                 "-",  color=color, linewidth=2, label=f"N={N} exact")
    axes[0].plot(h_values / J, data["vqe_energies"],
                 "o--", color=color, alpha=0.6, label=f"N={N} VQE")

    # Magnetisation comparison
    axes[1].plot(h_values / J, b["magnetisations"],
                 "-",  color=color, linewidth=2)
    axes[1].plot(h_values / J, data["magnetisations"],
                 "o--", color=color, alpha=0.6)

    # Energy gap
    axes[2].plot(h_values / J, b["gaps"],
                 "-", color=color, linewidth=2, label=f"N={N}")

axes[0].axvline(x=1.0, color="gray", linestyle=":")
axes[0].set_xlabel("h / J")
axes[0].set_ylabel("Ground state energy")
axes[0].set_title("Energy: VQE vs exact")
axes[0].legend(fontsize=7)

axes[1].axvline(x=1.0, color="gray", linestyle=":")
axes[1].set_xlabel("h / J")
axes[1].set_ylabel("|⟨M⟩|")
axes[1].set_title("Magnetisation: VQE (dashed) vs exact (solid)")

axes[2].axvline(x=1.0, color="gray", linestyle=":")
axes[2].set_xlabel("h / J")
axes[2].set_ylabel("Δ = E₁ - E₀")
axes[2].set_title("Energy gap (exact) — closes at critical point")
axes[2].legend()

plt.tight_layout()
plt.savefig("tfim_benchmarking.png", dpi=150)
plt.show()

In [ ]:
def find_pseudo_critical_point(N, h_values, benchmark):
    """
    Find pseudo-critical point using exact magnetisations.
    Restricts search to h/J in [0.7, 1.3] to avoid boundary artefacts.
    """
    mags   = benchmark[N]["magnetisations"]
    mask   = (h_values / J >= 0.7) & (h_values / J <= 1.3)
    h_sub  = h_values[mask]
    m_sub  = mags[mask]

    dM_dh  = np.gradient(m_sub, h_sub)
    idx    = np.argmin(dM_dh)
    return h_sub[idx]

pseudo_critical = {}
for N in N_values:
    h_c = find_pseudo_critical_point(N, h_values, benchmark)
    pseudo_critical[N] = h_c
    print(f"N={N:2d}  pseudo-critical point: h/J = {h_c:.3f}")

def scaling_form(N, h_c_inf, a):
    return h_c_inf + a * N ** (-1.0 / nu_exp)

N_arr   = np.array(list(pseudo_critical.keys()), dtype=float)
hc_arr  = np.array(list(pseudo_critical.values()))

popt, pcov = curve_fit(scaling_form, N_arr, hc_arr, p0=[1.0, 1.0])
h_c_inf, a = popt
h_c_err    = np.sqrt(pcov[0, 0])

print(f"\nExtrapolated critical point: h_c(∞) = {h_c_inf:.4f} ± {h_c_err:.4f}")
print(f"Exact answer:                h_c(∞) = 1.0000")
print(f"Error:                       {abs(h_c_inf - 1.0)*100:.2f}%")


plt.close('all')
fig = plt.figure(figsize=(18, 10))
fig.suptitle("TFIM Quantum Phase Transition — Full Summary", fontsize=14, fontweight="bold")

# Layout: 2 rows, 3 columns
ax1 = fig.add_subplot(2, 3, 1)   # Ground state energy
ax2 = fig.add_subplot(2, 3, 2)   # Magnetisation
ax3 = fig.add_subplot(2, 3, 3)   # Energy gap
ax4 = fig.add_subplot(2, 3, 4)   # Data collapse
ax5 = fig.add_subplot(2, 3, 5)   # VQE error
ax6 = fig.add_subplot(2, 3, 6)   # Finite-size extrapolation

colors = ["steelblue", "teal", "coral", "purple"]

for (N, data), color in zip(results.items(), colors):
    b         = benchmark[N]
    x         = h_values / J
    x_scaled  = (x - 1.0) * (N ** (1.0 / nu_exp))
    y_scaled  = b["magnetisations"] * (N ** (beta_exp / nu_exp))

    # 1. Ground state energy
    ax1.plot(x, b["energies"],        "-",   color=color, linewidth=2,  label=f"N={N} exact")
    ax1.plot(x, data["vqe_energies"], "o--", color=color, alpha=0.5)

    # 2. Magnetisation
    ax2.plot(x, b["magnetisations"],   "-",   color=color, linewidth=2, label=f"N={N}")
    ax2.plot(x, data["magnetisations"],"o--", color=color, alpha=0.5)

    # 3. Energy gap
    ax3.plot(x, b["gaps"], "-", color=color, linewidth=2, label=f"N={N}")

    # 4. Data collapse (exact)
    ax4.plot(x_scaled, y_scaled, "o-", color=color, label=f"N={N}")

    # 5. VQE error
    ax5.plot(x, np.abs(data["vqe_energies"] - b["energies"]),
             "o-", color=color, label=f"N={N}")

# 6. Finite-size extrapolation
N_fine  = np.linspace(3, 14, 100)
ax6.plot(N_arr, hc_arr, "o", color="steelblue", markersize=8, label="Pseudo-critical points")
ax6.plot(N_fine, scaling_form(N_fine, *popt), "--", color="coral",
         label=f"Fit: $h_c(∞)$ = {h_c_inf:.4f} ± {h_c_err:.4f}")
ax6.axhline(y=1.0, color="gray", linestyle=":", label="Exact $h_c$ = 1.0")
ax6.set_xlabel("N")
ax6.set_ylabel("$h_c(N)$ / J")
ax6.set_title("Finite-size extrapolation")
ax6.legend(fontsize=8)

# Formatting
for ax in [ax1, ax2, ax3, ax5]:
    ax.axvline(x=1.0, color="gray", linestyle=":", alpha=0.7)
    ax.set_xlabel("h / J")

ax4.axvline(x=0, color="gray", linestyle=":", alpha=0.7)
ax4.set_xlabel(r"$(h/J - 1) \cdot N^{1/\nu}$")
ax4.set_ylabel(r"$|\langle M \rangle| \cdot N^{\beta/\nu}$")
ax4.set_xlim(-8, 8)

ax1.set_ylabel("Ground state energy")
ax1.set_title("Ground state energy")
ax1.legend(fontsize=7)

ax2.set_ylabel("|⟨M⟩|")
ax2.set_title("Magnetisation: exact (solid) VQE (dashed)")
ax2.legend(fontsize=8)

ax3.set_ylabel("Δ = E₁ - E₀")
ax3.set_title("Energy gap")
ax3.legend(fontsize=8)

ax4.set_title("Finite-size scaling collapse")
ax4.legend(fontsize=8)

ax5.set_ylabel("|E_VQE - E_exact|")
ax5.set_title("VQE approximation error")
ax5.legend(fontsize=8)

plt.tight_layout()
plt.savefig("tfim_full_summary.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# VISUALISATION OF THE TFIM PHASE DIAGRAM 
# Plotting options
save_figures    = True               # save figures as PNG files
dpi             = 150                # resolution for saved figures
colors          = ["steelblue", "teal", "coral", "purple", "darkgreen", "orange"]  # one per N

# For heatmaps: custom colormap (blue -> white -> red)
cmap_phase      = LinearSegmentedColormap.from_list('phase', ['#1f77b4', 'white', '#d62728'], N=256)

# For phase boundary extrapolation (Figure 6)
nu_fit          = 1.0                # exponent used in fit (exact value = 1)

# Prepare data arrays 
x = h_values / J                      # reduced field

# Figure 1: Magnetisation (exact vs VQE) 
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for i, N in enumerate(N_values):
    color = colors[i % len(colors)]
    # Exact magnetisation
    ax1.plot(x, benchmark[N]['magnetisations'], 'o-', color=color,
             linewidth=2, markersize=4, label=f'N = {N}')
    # VQE magnetisation (dashed)
    ax2.plot(x, results[N]['magnetisations'], 'o--', color=color,
             linewidth=2, markersize=4, label=f'N = {N}')

for ax in (ax1, ax2):
    ax.axvline(x=1.0, color='gray', linestyle='--', linewidth=1.5, label='h/J = 1')
    ax.set_xlabel('h / J', fontsize=12)
    ax.set_ylabel(r'$\langle M \rangle$', fontsize=12)  
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
ax1.set_title('Exact magnetisation', fontsize=13)
ax2.set_title('VQE magnetisation', fontsize=13)
plt.tight_layout()
if save_figures:
    plt.savefig('TFIM_magnetisation.png', dpi=dpi, bbox_inches='tight')
plt.show()

# Figure 2: VQE energy error
plt.figure(figsize=(8, 5))
for i, N in enumerate(N_values):
    color = colors[i % len(colors)]
    error = np.abs(results[N]['vqe_energies'] - benchmark[N]['energies'])
    plt.semilogy(x, error, 'o-', color=color, linewidth=2, markersize=4, label=f'N = {N}')
plt.axvline(x=1.0, color='gray', linestyle='--', linewidth=1.5)
plt.xlabel('h / J', fontsize=12)
plt.ylabel(r'$|E_{\mathrm{VQE}} - E_{\mathrm{exact}}|$', fontsize=12)  
plt.title('VQE energy approximation error', fontsize=13)
plt.legend()
plt.grid(alpha=0.3, which='both')
plt.tight_layout()
if save_figures:
    plt.savefig('TFIM_VQE_error.png', dpi=dpi, bbox_inches='tight')
plt.show()

# Figure 3: Energy gap
plt.figure(figsize=(8, 5))
for i, N in enumerate(N_values):
    color = colors[i % len(colors)]
    plt.plot(x, benchmark[N]['gaps'], 'o-', color=color, linewidth=2, markersize=4, label=f'N = {N}')
plt.axvline(x=1.0, color='gray', linestyle='--', linewidth=1.5)
plt.xlabel('h / J', fontsize=12)
plt.ylabel(r'$\Delta = E_1 - E_0$', fontsize=12)  
plt.title('Energy gap closing at the quantum critical point', fontsize=13)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
if save_figures:
    plt.savefig('TFIM_energy_gap.png', dpi=dpi, bbox_inches='tight')
plt.show()

# Figure 4: Finite‑size scaling collapse 
plt.figure(figsize=(7, 5))
for i, N in enumerate(N_values):
    color = colors[i % len(colors)]
    x_scaled = (x - 1.0) * (N ** (1.0 / nu_exp))
    y_scaled = benchmark[N]['magnetisations'] * (N ** (beta_exp / nu_exp))
    plt.plot(x_scaled, y_scaled, 'o-', color=color, label=f'N = {N}')
plt.axvline(x=0, color='gray', linestyle='--', linewidth=1.5)
plt.xlabel(r'$(h/J - 1) \, N^{1/\nu}$', fontsize=12)      
plt.ylabel(r'$\langle M \rangle \, N^{\beta/\nu}$', fontsize=12)  
plt.title('Finite‑size scaling collapse (exact data)', fontsize=13)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
if save_figures:
    plt.savefig('TFIM_scaling_collapse.png', dpi=dpi, bbox_inches='tight')
plt.show()

# Figure 5: Heatmaps of magnetisation (exact and VQE) 
M_exact = np.array([benchmark[N]['magnetisations'] for N in N_values])
M_vqe   = np.array([results[N]['magnetisations'] for N in N_values])

fig, (ax5, ax6) = plt.subplots(1, 2, figsize=(12, 5))
extent = [x.min(), x.max(), N_values[0], N_values[-1]]

im1 = ax5.imshow(M_exact, aspect='auto', origin='lower', extent=extent,
                 cmap=cmap_phase, vmin=0, vmax=1)
ax5.axvline(x=1.0, color='black', linestyle='--', linewidth=1.5)
ax5.set_xlabel('h / J', fontsize=12)
ax5.set_ylabel('System size N', fontsize=12)
ax5.set_title('Exact magnetisation', fontsize=13)
cbar1 = plt.colorbar(im1, ax=ax5)
cbar1.set_label(r'$\langle M \rangle$', fontsize=11)   

im2 = ax6.imshow(M_vqe, aspect='auto', origin='lower', extent=extent,
                 cmap=cmap_phase, vmin=0, vmax=1)
ax6.axvline(x=1.0, color='black', linestyle='--', linewidth=1.5)
ax6.set_xlabel('h / J', fontsize=12)
ax6.set_ylabel('System size N', fontsize=12)
ax6.set_title('VQE magnetisation', fontsize=13)
cbar2 = plt.colorbar(im2, ax=ax6)
cbar2.set_label(r'$\langle M \rangle$', fontsize=11)  

plt.tight_layout()
if save_figures:
    plt.savefig('TFIM_phase_heatmap.png', dpi=dpi, bbox_inches='tight')
plt.show()

# Figure 6: Phase boundary shift with N (using your pseudo_critical dictionary) 
# Ensure pseudo_critical is a dict: {N: h_c}
if 'pseudo_critical' in globals():
    N_vals = np.array(sorted(pseudo_critical.keys()))
    hc_vals = np.array([pseudo_critical[N] for N in N_vals])
else:
    # Fallback: compute pseudo_critical from derivative (as you did earlier)
    # This code is already in your notebook; you can copy it here if needed.
    print("pseudo_critical not found; skipping Figure 6.")
    N_vals = np.array([])
    hc_vals = np.array([])

if len(N_vals) > 0:
    plt.figure(figsize=(8, 5))
    plt.errorbar(N_vals, hc_vals, yerr=np.zeros_like(hc_vals), fmt='o', capsize=4,
                 capthick=1, color='teal', label='Pseudo-critical point (derivative min)')
    plt.axhline(y=1.0, color='gray', linestyle='--', linewidth=1.5, label='Thermodynamic limit $h_c = 1$')
    plt.xlabel('System size $N$', fontsize=12)
    plt.ylabel('$h_c(N) / J$', fontsize=12)
    plt.title('Finite‑size shift of the phase boundary', fontsize=13)
    plt.legend()
    plt.grid(alpha=0.3)

    # Fit scaling law: h_c(N) = h_c(∞) + a * N^{-1/ν}
    def scaling_law(N, h_inf, a):
        return h_inf + a * N ** (-1.0 / nu_fit)

    popt, pcov = curve_fit(scaling_law, N_vals, hc_vals, p0=[1.0, 1.0])
    N_fine = np.linspace(min(N_vals), max(N_vals), 100)
    plt.plot(N_fine, scaling_law(N_fine, *popt), '--', color='coral',
             label=f'Fit: $h_c(\\infty) = {popt[0]:.3f} \\pm {np.sqrt(pcov[0,0]):.3f}$')
    plt.legend()
    plt.tight_layout()
    if save_figures:
        plt.savefig('TFIM_phase_boundary_vs_N.png', dpi=dpi, bbox_inches='tight')
    plt.show()

In [ ]:
def exact_entanglement_entropy(N: int, J: float, h: float,
                                subsystem_size: int = None) -> float:
    """
    Compute von Neumann entanglement entropy of the ground state.
    Uses epsilon=0 to avoid symmetry-breaking inflation for small N.
    """
    if subsystem_size is None:
        subsystem_size = N // 2

    # epsilon=0 here — symmetry breaking artificially inflates entanglement for small N
    H_matrix             = build_tfim_hamiltonian(N, J, h, epsilon=0.0).to_matrix()
    eigenvalues, eigvecs = np.linalg.eigh(H_matrix)
    ground_state         = np.ascontiguousarray(eigvecs[:, 0])

    psi             = ground_state.reshape(2**subsystem_size,
                                           2**(N - subsystem_size))
    singular_values = np.linalg.svd(psi, compute_uv=False)
    schmidt_sq      = singular_values**2
    schmidt_sq      = schmidt_sq[schmidt_sq > 1e-12]
    return float(-np.sum(schmidt_sq * np.log(schmidt_sq)))

# Sweep entropy across phase diagram for all N
entropy_results = {}
for N in N_values:
    print(f"Computing entanglement entropy for N={N}...")
    entropies = [exact_entanglement_entropy(N, J, h) for h in h_values]
    entropy_results[N] = np.array(entropies)

# Plot
plt.close('all')
fig, ax = plt.subplots(figsize=(8, 5))
for N, color in zip(N_values, colors):
    ax.plot(h_values / J, entropy_results[N], "o-",
            label=f"N={N}", color=color)
ax.axvline(x=1.0, color="gray", linestyle=":", label="Critical point h/J=1")
ax.set_xlabel("h / J")
ax.set_ylabel("Entanglement entropy S")
ax.set_title("Von Neumann entanglement entropy across TFIM phase diagram")
ax.legend()
plt.tight_layout()
plt.savefig("tfim_entanglement_entropy.png", dpi=150)
plt.show()

In [ ]:
for N in N_values:
    M = exact_magnetisation(N, J, h=0.2)
    print(f"N={N:2d}  |M| = {M:.6f}")   # all should be close to 1.0